# Loop anomalies

This notebook investigates why we've seen anormalities in Loop that Lane discovered:
 - has a wider range of egv (i.e. not 40-400 like all the rest)
 - has a tail of very small iob's (the rest don't)
 - has a tail of very small tdd's (the rest don't)

In [ ]:
import os, sys
%matplotlib widget
import matplotlib.pyplot as plt
sys.path.append(os.path.join(os.getcwd(),'..','..'))
from datetime import timedelta,datetime
from src import cdf
from src import pandas_helper, drawing
from studies import dataset_initializer
import random 
random.seed(42)  # For reproducibility
import pandas as pd
import numpy as np
from src import tdd

In [ ]:
data_path = os.path.join(os.getcwd(), '..','..','data', 'raw')
studies = dataset_initializer.initialize_datasets(data_path)
loop = studies['Loop']

In [ ]:
loop.load_data()

## CGM

In [ ]:
df_cgm = loop.extract_cgm_history()

In [ ]:
df_low = df_cgm.loc[df_cgm.cgm<=45]
df_high = df_cgm.loc[df_cgm.cgm>=395]


fig, AX = plt.subplots(1, 2, figsize=(12, 3))
cdf.plot_cdf(df_low.cgm, log_scaled=True, ax=AX[0])
AX[0].hist(df_low.cgm,histtype='step', color='blue')
AX[0].set_title('CGM Values below 45 mg/dL')
AX[0].set_xlabel('CGM Value (mg/dL)')
AX[0].set_ylabel('CDF')

cdf.plot_cdf(df_high.cgm, log_scaled=True, ax=AX[1])
AX[1].hist(df_high.cgm,histtype='step', color='blue')
AX[1].set_title('CGM Values above or equal to 400 mg/dL')
AX[1].set_xlabel('CGM Value (mg/dL)')
AX[1].set_ylabel('CDF')

plt.tight_layout()
plt.show()

#plt.savefig(os.path.join(os.getcwd(), '..', '..', 'docs/data_sets/assets/loop_cgm_anomalies.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(os.getcwd(), '..', '..', 'docs/data_sets/assets/loop_cgm_anomalies.png'), bbox_inches='tight', dpi=300)

In [ ]:
from src import pandas_helper
temp = df_high.loc[df_high.cgm.between(400, 405)].cgm.value_counts().sort_values(ascending=False)
display(temp)

temp = df_low.loc[df_low.cgm.between(35, 40)].cgm.value_counts().sort_values(ascending=False)
display(temp)

In [ ]:
df_anomal = df_cgm.loc[(df_cgm.cgm < 38) | (df_cgm.cgm > 401.06)].copy()
print(f'Overall there are {df_anomal.patient_id.nunique()} patients with CGM values below 38 or above 401.06 mg/dL in the dataset.')
print(f'Overall there are {df_anomal.shape[0]} CGM values below 38 or above 401.06 mg/dL in the dataset that is {df_anomal.shape[0]/df_cgm.shape[0]:.6%}% of all cgm data')

In [ ]:
(df_cgm.cgm <= 38).sum(), (df_cgm.cgm > 401.06).sum()

So there are only 4 values below 38 and 34 above 401.06

Now, for these patients, let's draw example days with values containing their minimum values:

In [ ]:
#draw examples for different anomal CGM  values
n=3
sample_dict = {
    'lt_35': df_cgm[df_cgm['cgm'] < 35].sample(n),
    '35_40': df_cgm[(df_cgm['cgm'] > 38) & (df_cgm['cgm'] < 40)].sample(n),
    '400_402': df_cgm[(df_cgm['cgm'] > 400) & (df_cgm['cgm'] < 401.06)].sample(n),
    'gt_405': df_cgm[df_cgm['cgm'] > 401.06].sample(n)}
display(sample_dict)

#draw
f, AX = plt.subplots(4,n, figsize=(15, n*2.75),sharey=True)
for i,(key,rows) in enumerate(sample_dict.items()):
    for j,(index, row) in enumerate(rows.iterrows()):
        ax = AX[i,j] 
        sub_frame = df_cgm.loc[(df_cgm.patient_id==row.patient_id) & (df_cgm.datetime.between(row.datetime - timedelta(hours=6), row.datetime + timedelta(hours=6)))]
        drawing.drawCGM(ax, sub_frame.datetime.values, sub_frame.cgm.values)
        
        ax.axhline(40, color='gray', linestyle='--', linewidth=1)
        ax.axhline(400, color='gray', linestyle='--', linewidth=1) 
        
        # Highlight the extreme value
        ax.scatter([row.datetime], [row.cgm], color='red', marker='.', s=100, facecolor='none', edgecolor='red')
        ax.text(row.datetime, row.cgm + 10, f"{row.cgm:.2f}",color='red',ha='left',va='bottom',fontsize=12)
        ax.set_title(f'Patient {row.patient_id} - {key}')
        drawing.format_time_axis(ax) 
        plt.tight_layout()

In [ ]:
#plt.savefig(os.path.join(os.getcwd(), '..', '..', 'docs/data_sets/assets/loop_anormalities.pdf'), bbox_inches='tight')

### Observations CGM

- Values of ~38 to ~39 and around ~401 reflect out of range values --> **replace with to 40, 400**
 - Values have numerical inaccuracies (e.g. 39.005186 instead of 39)

- only 40 values > 401.6
 - User 613 seems to be using a CGM that reports glucose > 400 --> **keep**
 - 3 users with values of 1,10,20
  - probably during sensor erroneous --> **should be removed**

## Bolus

In [ ]:
df_bolus = loop._extract_bolus_event_history()
df_bolus['date'] = df_bolus.datetime.dt.date

#### CDF of daily bolus amounts

In [ ]:
# Plotting the CDF of daily bolus amounts
plt.figure(figsize=(10, 3));ax = plt.gca()
cdf.plot_cdf(df_bolus.groupby(['patient_id','date']).bolus.sum(), 'Daily Bolus Amounts', 'Bolus Amount (Units)', 'Days',ax=ax, log_scaled=True,percent_right_axis=True)
plt.tight_layout()
plt.xlim(-.01,1.1)

In [ ]:
#How many patients are there that have days with < 1U bolus 
n = df_bolus.groupby(['patient_id','date']).bolus.sum().reset_index().groupby('patient_id').apply(lambda x: (x.bolus < 1).any(),include_groups=False).sum()
print(f'There are {n} patients that have days with < 1U bolus')

 - ~ 0.5% of days with <1U of total boluses.
 - But 361 patients have at least one day with <1 U 

Now, let's look at some examples of such days:

In [ ]:
# Let's take a look at some days with the smallest amounts of boluses and surrounding days
daily_bolus_sum = df_bolus.groupby(['patient_id','date']).bolus.sum().sort_values(ascending=False).reset_index()
display(daily_bolus_sum.tail(10))

# Draw data for all patients in the last 10 items of the list
tail_samples = daily_bolus_sum.tail(5)
for idx, sample in tail_samples.iterrows():
    #print(f"Patient {sample.patient_id}, Day {sample.day}, Bolus {sample.bolus}")
    #display(sample)
    #display(df_bolus.set_index(['patient_id','date']).loc[(sample.patient_id, sample.day)])

    day = datetime.combine(sample.date, datetime.min.time())
    patient = sample.patient_id
    date_range = (day - timedelta(days=10), day + timedelta(days=10))
    temp = df_bolus.loc[
        (df_bolus.patient_id == patient) &
        df_bolus.datetime.between(date_range[0], date_range[1])
    ]
    zero_boluses = temp.loc[temp.bolus == 0]

    plt.figure(figsize=(12, 2))

    # Highlight the background for the selected day
    ax = plt.gca()
    highlight_start = day
    highlight_end = day + timedelta(days=1)
    ax.axvspan(highlight_start, highlight_end, color='lightgray', alpha=0.3, zorder=0)
    
    #add day separator lines
    for dt in pd.date_range(temp.datetime.min().normalize(), temp.datetime.max().normalize()):
        ax.axvline(dt, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    
    # Draw the boluses
    drawing.drawBoluses(ax, temp.datetime, temp.bolus)
    ax.scatter(zero_boluses.datetime, zero_boluses.bolus, color='blue', s=10, label='Zero Bolus', alpha=0.5,zorder=4)
    drawing.format_time_axis(ax, minor_interval_hours=6)
    
    plt.ylim(-0.1, None)
    plt.xlim(date_range)
    plt.title(f'Patient {patient} - Day {sample.date}')
    plt.tight_layout()
    plt.show()

- The selected days mostly only have a single 0.05 unit bolus

By looking at the surrounding days we see that:
- Some patients show days without boluses like as if these days are missing (982,68)
- Some patients have days with boluses of zero value (66), maybe cancelled boluses
- Some patients only seem to bolus every now and then (or missing?) (569,67)
- Some patients show only micro boluses (1183)


#### Bolus statistics

In [ ]:
#calculate patient bolus statistics
def p25(x):
    return np.percentile(x, 25)
def p75(x):
    return np.percentile(x, 75)

df_daily_bolus_stats = df_bolus.groupby(['patient_id','date']).agg({'bolus':['sum','count']}).bolus.groupby('patient_id').agg(['min', p25,p75, 'mean', 'max', 'sum'])
display(df_daily_bolus_stats.sample(5))


In [ ]:
# CDFs in a 2x2 subplot grid

fig, axs = plt.subplots(2, 2, figsize=(10, 5))

# Average events / day
cdf.plot_cdf(df_daily_bolus_stats['count']['mean'], 'Average Bolus Events per Day', 'Bolus Events', 'Patients', ax=axs[0, 0])
axs[0, 0].set_xscale('log')
axs[0, 0].set_yscale('log')

# Average total bolus dose per day
cdf.plot_cdf(df_daily_bolus_stats['sum']['mean'], 'Average Total Bolus Dose per Day', 'Bolus Dose (U)', 'Patients', ax=axs[0, 1])
axs[0, 1].set_xscale('log')
axs[0, 1].set_yscale('log')

# Days per patient
cdf.plot_cdf(df_bolus.groupby('patient_id').date.nunique(), 'Days with Bolus Data per Patient', 'Days', 'Patients', ax=axs[1, 0])
axs[1, 0].set_xscale('log')
axs[1, 0].set_yscale('log')

# p75 - p25 of average bolus dose per day
cdf.plot_cdf(
    df_daily_bolus_stats['sum']['p75'] - df_daily_bolus_stats['sum']['p25'],
    'Difference between 75th and 25th Percentile of Average Bolus Dose per Day',
    'Dose Difference (p75-p25) (U)', 'Patients', ax=axs[1, 1]
)
axs[1, 1].set_xscale('log')
axs[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()

We know that 0.5% of days have <1 U/day

Now we learned that there are 
 - ~ n=10 patients with < 2.5 bolus events / day in average
 - ~ n=30 patients with < 5 U/day in average
 - ~ 1% patients with < 10 days of data
 - ~ 5% with >20U in p75-p25 daily dose (could indicate missing data)
 - << with almost zero avg daily boluses
 
In summary: Very few patients seem to have very bolus events, average boluses or days. 
We also know that only 0.5% of days 

In [ ]:
# Boxplot of daily total bolus dose per patient (horizontal)
draw = False
if draw:
    import matplotlib.pyplot as plt

    # Prepare data: group by patient and day, sum bolus per day
    bolus_per_day = df_bolus.groupby(['patient_id', 'date']).bolus.sum().reset_index()

    # Sort patient_ids by mean_doses (ascending)
    mean_doses_sorted = bolus_per_day.groupby('patient_id').bolus.median().sort_values().index

    # Pivot to get a list of daily doses per patient
    data = bolus_per_day.groupby('patient_id').bolus.apply(lambda x: x.values).loc[mean_doses_sorted].reset_index()
    labels = data.patient_id.astype(str)

    plt.figure(figsize=(10, max(4, len(labels) * 0.1)))
    plt.boxplot(data.bolus.values.tolist(), vert=False, labels=labels.values, showfliers=False)
    plt.xlabel('Daily Total Bolus Dose (U)')
    plt.ylabel('Patient ID')
    plt.title('Daily Total Bolus Dose per Patient')
    plt.tight_layout()

    plt.axvline(1, color='orange', linestyle='--', linewidth=1, label='1U')

    # Highlight patients with minimum daily bolus dose of zero
    for i, vals in enumerate(data.bolus.values):
        min_val = np.min(vals)
        if min_val == 0:
            plt.scatter(min_val, i + 1, color='red', s=10, zorder=3, label='Min=0' if i == 0 else "",alpha=0.5)

    # Only show one legend entry for 'Min=0'
    handles, labels_ = plt.gca().get_legend_handles_labels()
    if 'Min=0' in labels_:
        plt.legend(loc='lower right')
    plt.show()

    #save as pdf
    #plt.savefig(os.path.join(os.getcwd(), '..', '..', 'docs/data_sets/assets/loop_daily_bolus_dose_boxes.pdf'), bbox_inches='tight')

## TDDs

In [ ]:
df_basal = loop.extract_basal_event_history()
df_basal['date'] = df_basal['datetime'].dt.date

# calculate tdds
from src import tdd
tdd_bolus = df_bolus.groupby(['patient_id'],observed=False).apply(lambda x: tdd.calculate_daily_bolus_dose(x), include_groups=False)
tdd_basal = df_basal.groupby(['patient_id'],observed=False).apply(lambda x: tdd.calculate_daily_basal_dose(x), include_groups=False)

# Count daily bolus and basal events
bolus_counts = df_bolus.groupby(['patient_id', 'date']).size().rename('bolus_count')
basal_counts = df_basal.groupby(['patient_id', 'date']).size().rename('basal_count')

#merge counts and tdds
tdd_basal = tdd_basal.merge(basal_counts, on=['patient_id', 'date'], how='left')
tdd_bolus = tdd_bolus.merge(bolus_counts, on=['patient_id', 'date'], how='left')

#combine in tdd dataframe
tdds = pd.merge(tdd_bolus, tdd_basal, how='outer', on=['patient_id','date'], suffixes=('_bolus', '_basal')).reset_index()
tdds['total']= tdds['bolus'] + tdds['basal']
display(tdds.sample(4))

In [ ]:
from importlib import reload
reload(cdf)
#CDF of TDDs
from matplotlib.ticker import FuncFormatter
#plot tdd cdfs
plt.figure(figsize=(11, 4)); ax= plt.gca()
cdf.plot_cdf(tdds.basal.dropna(), ax=ax,log_scaled=True,label='Basal TDD',percent_right_axis=True)
cdf.plot_cdf(tdds.bolus.dropna(), ax=ax,log_scaled=True,label='Bolus TDD',percent_right_axis=True,color='orange')
cdf.plot_cdf(tdds['total'].dropna(), ax=ax,log_scaled=True,label='Total TDD',percent_right_axis=True,color='blue')
ax.legend()
plt.xscale('log')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.4f}'))
plt.tight_layout()

 - 0.01% with TDD < 1U
 - 0.05% with TDD < 3U

In [ ]:
#stats
def mingezero(x):
    """Calculate the minimum TDD greater than zero."""
    return x.loc[x > 0].min()

def count_zeros(x):
    """Count the number of zeros."""
    return (x == 0).sum()

def count_nans(x):
    """Count the number of NaNs."""
    return x.isna().sum()

tdds.agg({
    'basal': ['min', mingezero, 'mean', 'max', count_zeros, count_nans],
    'bolus': ['min', mingezero, 'mean', 'max', count_zeros, count_nans],
    'total':   ['min', mingezero, 'mean', 'max', count_zeros, count_nans]
})

 - We see some 1700 days with NaN basal TDD: basal tdd set to nan when insufficient support points (assumption pump reports at least once when flat basal)
 - Minimum TDD is 0.15U/day. Lane reported minimum TDD in Loop of 0.0014 Units which is not what we see. 

#### Example days with low TDDs and surrounding data

In [ ]:
# Select random TDDs with sum < 1 and plot bolus and basal data around those days
from src import drawing

# Filter for days with sum < 1
low_tdd_days = tdds.loc[tdds['total'] < 1].sample(5, random_state=42)

for idx, row in low_tdd_days.iterrows():
    patient = row['patient_id']
    day = row['date']
    day_dt = pd.to_datetime(day)
    date_range = (day_dt - timedelta(days=5), day_dt + timedelta(days=5))
    
    # Bolus data for this patient and range
    temp_bolus = df_bolus.loc[
        (df_bolus.patient_id == patient) &
        (df_bolus.datetime.between(date_range[0], date_range[1]))
    ]
    # Basal data for this patient and range
    temp_basal = df_basal.loc[
        (df_basal.patient_id == patient) &
        (df_basal.datetime.between(date_range[0], date_range[1]))
    ]
    
    plt.figure(figsize=(14, 3))
    ax = plt.gca()
    # Highlight the background for the selected day
    highlight_start = day_dt
    highlight_end = day_dt + timedelta(days=1)
    ax.axvspan(highlight_start, highlight_end, color='lightgray', alpha=0.3, zorder=0)
    
    # Add day separator lines
    for dt in pd.date_range(date_range[0].normalize(), date_range[1].normalize()):
        ax.axvline(dt, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    
    # Draw boluses
    drawing.drawBoluses(ax, temp_bolus.datetime, temp_bolus.bolus)
    ax.set_ylabel('Bolus (U)', color='orange')
    
    # Draw basal rates
    if not temp_basal.empty:
        axtwin = ax.twinx()
        drawing.drawBasal(axtwin, temp_basal.datetime.values, temp_basal.basal_rate.values)
        axtwin.set_ylabel('Basal Rate (U/h)', color='blue')
    
    drawing.format_time_axis(ax, minor_interval_hours=6)

    # Add text annotation with bolus, basal, and total TDD for the selected day
    for dt in pd.date_range(date_range[0], date_range[1]):
        tdd_row = tdds[(tdds['patient_id'] == patient) & (tdds['date'] == dt.date())]
        if not tdd_row.empty:
            ax.text(
                dt.normalize(),
                ax.get_ylim()[1] * 0.8, 
                f"Bolus: {tdd_row['bolus'].values[0]:.2f} U\nBasal: {tdd_row['basal'].values[0]:.2f} U\nTotal: {tdd_row['total'].values[0]:.2f} U", color='black', fontsize='x-small', bbox=dict(facecolor='white', alpha=0.7))
    plt.xlim(date_range)
    plt.title(f'Patient {patient} - Day {day} (TDD < 1U)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Examples for lowest basal TDDs
from src import drawing
low_tdd_days = tdds.sort_values(by='basal',ascending=False).dropna().tail(20).sample(5)

for idx, row in low_tdd_days.iterrows():
    patient = row['patient_id']
    day = row['date']
    day_dt = pd.to_datetime(day)
    date_range = (day_dt - timedelta(days=10), day_dt + timedelta(days=10))
    
    # Bolus data for this patient and range
    temp_bolus = df_bolus.loc[
        (df_bolus.patient_id == patient) &
        (df_bolus.datetime.between(date_range[0], date_range[1]))
    ]
    # Basal data for this patient and range
    temp_basal = df_basal.loc[
        (df_basal.patient_id == patient) &
        (df_basal.datetime.between(date_range[0], date_range[1]))
    ]
    
    plt.figure(figsize=(14, 3))
    ax = plt.gca(); 

    # Draw basal rates
    if not temp_basal.empty:
        drawing.drawBasal(ax, temp_basal.datetime.values, temp_basal.basal_rate.values)
        ax.set_ylabel('Basal Rate (U/h)', color='blue')

    # Highlight the background for the selected day
    highlight_start = day_dt
    highlight_end = day_dt + timedelta(days=1)
    ax.axvspan(highlight_start, highlight_end, color='lightgray', alpha=0.3, zorder=0)
    
    # Add day separator lines
    for dt in pd.date_range(date_range[0].normalize(), date_range[1].normalize()):
        ax.axvline(dt, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    
    drawing.format_time_axis(ax, minor_interval_hours=6)

    # Draw boluses
    twinx = ax.twinx()
    drawing.drawBoluses(twinx, temp_bolus.datetime, temp_bolus.bolus)
    twinx.set_ylabel('Bolus (U)', color='orange')
    

    # Add text annotation with bolus, basal, and sum TDD for the selected day
    for dt in pd.date_range(date_range[0], date_range[1]):
        tdd_row = tdds[(tdds['patient_id'] == patient) & (tdds['date'] == dt.date())]
        if not tdd_row.empty:
            ax.text(
                dt.normalize(),
                ax.get_ylim()[1] * 0.8, 
                f"Bolus: {tdd_row['bolus'].values[0]:.2f} U\nBasal: {tdd_row['basal'].values[0]:.2f} U\n total: {tdd_row['total'].values[0]:.2f} U", color='black', fontsize='x-small', bbox=dict(facecolor='white', alpha=0.7))
    plt.xlim(date_range)
    plt.title(f'Patient {patient} - Day {day}')
    plt.tight_layout()
    plt.show()

We know that only in very rare cases TDDs are extremely low. 
It appears af if the majority of these cases is due to missing data.
Mostly, basal data is missing. Unclear why.

## Inspecting Tail end of median TDDs

Despite the extreme low end values, we also see that there are many TDDs with comparably small values. Let's have alook: 

In [ ]:
median_tdds = tdds.groupby('patient_id').agg({'total': 'median'}).reset_index().rename(columns={'total': 'median_tdd'}).sort_values('median_tdd', ascending=False).dropna()
lowest_median_tdds = median_tdds.tail(10)
lowest_median_tdds

### Minors?

Tdds are so low, are these minors?

In [ ]:
#display the paitent info for the patients with lowest median TDDs
loop._df_patient.loc[loop._df_patient.PtID.isin(lowest_median_tdds.patient_id.astype(int).values)]

- Most are minors

In [ ]:
## let's look at the age distribution of all patients
cdf.plot_cdf(loop._df_patient.AgeAtEnrollment)

### Patient 1183
Let's lok at 1183 (age 31) - Is this patient in a honey moon phase?

In [ ]:
#tdd for patient 1183
plt.figure(figsize=(8,3)); ax = plt.gca();
cdf.plot_cdf(tdds.loc[tdds.patient_id == '1183'].bolus,ax=ax,color='orange',label='Bolus TDD')
cdf.plot_cdf(tdds.loc[tdds.patient_id == '1183'].basal,ax=ax,color='blue',label='Basal TDD')
cdf.plot_cdf(tdds.loc[tdds.patient_id == '1183']['total'],ax=ax, label='Total TDD')
ax.set_title('TDD CDF for Patient 1183')
ax.legend()
plt.show()

#tdds over time
plt.figure(figsize=(8,3))
temp = tdds.loc[tdds.patient_id == '1183']
plt.scatter(temp.date,temp['total'],s=10,color='gray')
plt.scatter(temp.date,temp['bolus'],s=10, color='orange')
plt.scatter(temp.date,temp['basal'],s=10, color='blue')
plt.xlabel('Date');
plt.ylabel('TDD (U)');
plt.title('TDD Over Time for Patient 1183');
plt.show()

In [ ]:
# Draw CGM data for patient 1183
patient_id = '1183'
df_cgm_patient = df_cgm.loc[df_cgm.patient_id == patient_id].copy()

tir = df_cgm_patient.cgm.agg(lambda x: ((x>=70) & (x<=180)).mean())
print(f'TIR for patient {patient_id}: {tir:.2%}')

def get_random_cgm_day(df, seed=None):
    df['datetime'] = pd.to_datetime(df['datetime'])
    if seed is not None:
        np.random.seed(seed)
    random_date = np.random.choice(df['datetime'].dt.date.unique())
    return df[df['datetime'].dt.date == random_date]

plt.figure(figsize=(7, 2.5)); ax = plt.gca()

for i in range(10):
    random_day = get_random_cgm_day(df_cgm_patient, seed=i)    
    ax.scatter((random_day.datetime-random_day.datetime.min().normalize()).apply(lambda x: x.total_seconds() / 3600), random_day.cgm,label=f'day={random_day.datetime.dt.date.values[0]}',
                s=1, alpha=0.5, zorder=1)
    ax.axhspan(70, 180, color='lightgreen', alpha=0.2, zorder=0)
    
plt.ylim(0,300)
plt.xlabel('Hour of day',fontdict={'fontsize': 'x-small'})
plt.title(f'Example Days for Patient 1183',fontsize='x-small')
plt.tight_layout()
#plt.savefig(os.path.join(os.getcwd(), '..', '..', 'docs/data_sets/assets/loop_cgm_patient_1183.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(os.getcwd(), '..', '..', 'docs/data_sets/assets/loop_cgm_patient_1183.png'), bbox_inches='tight', dpi=300)

### age vs. median tdd

In [ ]:
ages = loop._df_patient[['PtID','AgeAtEnrollment']].copy()
ages['PtID'] = ages['PtID'].astype(str)
mediantdds_age = median_tdds.merge(ages, left_on='patient_id', right_on='PtID', how='left')
mediantdds_age

#scatter plot
plt.figure(figsize=(5, 3)); ax = plt.gca()
ax.scatter(mediantdds_age['AgeAtEnrollment'], mediantdds_age['median_tdd'],alpha=0.25)
plt.xlabel('Age at Enrollment')
plt.ylabel('Median TDD')
plt.title('Age vs TDD')
plt.xlim(0,40)
plt.ylim(0, 100)
plt.tight_layout()
ax.tick_params(axis='both', labelsize='x-small')
#ax.title.set_fontsize('x-small')
ax.xaxis.label.set_fontsize('x-small')
ax.yaxis.label.set_fontsize('x-small')

plt.savefig(os.path.join(os.getcwd(), '..', '..', 'docs/data_sets/assets/loop_age_vs_tdd.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(os.getcwd(), '..', '..', 'docs/data_sets/assets/loop_age_vs_tdd.png'), bbox_inches='tight', dpi=300)

## In summary

**EGV**:  
- Values of ~38 to ~39 and around ~401 reflect out of range values --> **replace with to 40, 400**
 - Values have numerical inaccuracies (e.g. 39.005186 instead of 39)
- only 6 values < 38 and 34 values > 401.06
 - User 613 seems to be using a CGM that reports glucose > 400 --> **keep**
 - 3 users with values of 1,10,20
  - probably during sensor erroneous --> **should be removed**

**TDDs**: 
 - Small TDDs are linked to minors or patients in honeymoon phase
 - extreme low TDDs (basal, bolus, or both) are most likely missing data
  - these often occur when data is missing in neighboring days too
  - often we see only boluses and no basal rates reported
  - Sometimes we see singular micro boluses 
 - There is **no general rule** that would apply. One solution could be to exclude days with less than n basal rates and no boluses.